# Session 7: MongoDB Indexing

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-07_index_types.py`

In [2]:
# Session 7 · Creating the main index types
from pymongo import ASCENDING, DESCENDING
from pymongo.errors import DuplicateKeyError
from store_db import get_db

db = get_db()

made = [
    db.products.create_index("category"),                                         # single field
    db.products.create_index([("category", ASCENDING), ("price", DESCENDING)]),    # compound
    db.products.create_index("tags"),                                             # multikey (array field)
    db.products.create_index("specs.brand", name="brand_idx"),                    # embedded field, custom name
    db.orders.create_index([("customerId", 1), ("orderDate", -1)]),               # supports "my latest orders"
    db.customers.create_index("joined", expireAfterSeconds=60 * 60 * 24 * 365),   # TTL (on a date field)
]
print("created:", made)

print("\nproducts indexes:")
for name, info in db.products.index_information().items():
    print(f"  {name:<22} key={info['key']}")

print("\nUnique index on usernames")
db.users.create_index("username", unique=True)
db.users.insert_one({"username": "ayesha.k"})
try:
    db.users.insert_one({"username": "ayesha.k"})
except DuplicateKeyError:
    print("  DuplicateKeyError: username 'ayesha.k' is taken")

print("\nDropping an index")
db.products.drop_index("brand_idx")
print("  remaining:", list(db.products.index_information()))

created: ['category_1', 'category_1_price_-1', 'tags_1', 'brand_idx', 'customerId_1_orderDate_-1', 'joined_1']

products indexes:
  _id_                   key=[('_id', 1)]
  category_1             key=[('category', 1)]
  category_1_price_-1    key=[('category', 1), ('price', -1)]
  tags_1                 key=[('tags', 1)]
  brand_idx              key=[('specs.brand', 1)]

Unique index on usernames
  DuplicateKeyError: username 'ayesha.k' is taken

Dropping an index
  remaining: ['_id_', 'category_1', 'category_1_price_-1', 'tags_1']


## Example: `session-07_scan_vs_index.py`

In [3]:
# Session 7 · Why indexes matter: a collection scan versus an index scan
# mongomock has no query planner, so this file models what MongoDB does with
# 200,000 synthetic orders and reports the same counters you see in explain():
# totalDocsExamined, totalKeysExamined and nReturned.
import bisect, random, time

random.seed(7)
N = 200_000
cities = ["Karachi", "Lahore", "Islamabad", "Peshawar", "Quetta", "Multan"]
orders = [{"_id": i, "customerId": random.randint(1, 20_000),
           "city": random.choice(cities), "amount": random.randint(500, 300_000)} for i in range(N)]

def collscan(pred):
    t = time.perf_counter()
    out = [d for d in orders if pred(d)]
    return out, {"stage": "COLLSCAN", "docsExamined": N, "keysExamined": 0,
                 "nReturned": len(out), "ms": round((time.perf_counter() - t) * 1000, 1)}

# Build a B-tree-like index on customerId: sorted (key, position) pairs
index = sorted((d["customerId"], pos) for pos, d in enumerate(orders))
keys = [k for k, _ in index]

def ixscan(value):
    t = time.perf_counter()
    lo, hi = bisect.bisect_left(keys, value), bisect.bisect_right(keys, value)
    out = [orders[index[i][1]] for i in range(lo, hi)]
    return out, {"stage": "IXSCAN -> FETCH", "docsExamined": len(out), "keysExamined": hi - lo,
                 "nReturned": len(out), "ms": round((time.perf_counter() - t) * 1000, 3)}

target = 4242
_, s1 = collscan(lambda d: d["customerId"] == target)
_, s2 = ixscan(target)
print(f"Query: find({{customerId: {target}}}) on {N:,} orders\n")
for s in (s1, s2):
    print(f"  {s['stage']:<16} docsExamined={s['docsExamined']:>7,}  keysExamined={s['keysExamined']:>3}  "
          f"nReturned={s['nReturned']}")
print("\n  Selectivity: the index read", s2["keysExamined"], "keys instead of", f"{N:,}", "documents.")
print("  Timing on this machine: collection scan is roughly",
      max(1, round(s1["ms"] / max(s2["ms"], 0.001))), "times slower (exact numbers vary per run).")

# Covered query: everything needed is in the index, so no FETCH at all
covered = [k for k in keys[bisect.bisect_left(keys, 100):bisect.bisect_right(keys, 105)]]
print(f"\nCovered query find({{customerId: {{$gte: 100, $lte: 105}}}}, {{_id: 0, customerId: 1}}):")
print(f"  keysExamined={len(covered)}  docsExamined=0  (answered from the index alone)")

# Low selectivity: an index on city still touches 1/6 of the collection
per_city = sum(1 for d in orders if d["city"] == "Karachi")
print(f"\nLow-selectivity field: city = 'Karachi' matches {per_city:,} of {N:,} "
      f"({per_city / N:.0%}). An index helps far less here.")

Query: find({customerId: 4242}) on 200,000 orders

  COLLSCAN         docsExamined=200,000  keysExamined=  0  nReturned=7
  IXSCAN -> FETCH  docsExamined=      7  keysExamined=  7  nReturned=7

  Selectivity: the index read 7 keys instead of 200,000 documents.
  Timing on this machine: collection scan is roughly 442 times slower (exact numbers vary per run).

Covered query find({customerId: {$gte: 100, $lte: 105}}, {_id: 0, customerId: 1}):
  keysExamined=55  docsExamined=0  (answered from the index alone)

Low-selectivity field: city = 'Karachi' matches 33,215 of 200,000 (17%). An index helps far less here.
